# GIF Transition Pipeline
Generate numbered transition gifs, review them, then display all together.

## Setup

## Config


In [ ]:
from PIL import Image
import os
import torch
import numpy as np
import random
from diffusers import StableDiffusionImg2ImgPipeline
from diffusers.utils import load_image
from IPython.display import display, HTML, Image as IPyImage
import base64

device = "cuda"
print(f"Using device: {device}")

# --- Output paths ---
GIFS_DIR = r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\stabilityaiturboxd_pipeline\Calder\outputs\gifs"
os.makedirs(GIFS_DIR, exist_ok=True)

# ┌─────────────────────────────────────────────────────────────────────┐
# │  THEME FOLDERS — add your folder paths here, one entry per theme.  │
# │  Key   = any label you want (used in sequence log + shuffle logic) │
# │  Value = path to the folder containing that theme's images         │
# └─────────────────────────────────────────────────────────────────────┘
THEME_DIRS = {
    "SUBSTRATE": r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\IMAGE STACK\SUBSTRATE  MATERIAL WORLD",   # ← REPLACE with your folder path
    "MACHINE": r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\IMAGE STACK\MACHINE",   # ← REPLACE with your folder path
    "CIRCULATION": r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\IMAGE STACK\CIRCULATION",   # ← REPLACE with your folder path
    "REPRESENTATION": r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\IMAGE STACK\REPRESENTATION",   # ← REPLACE with your folder path
    "CONTROL": r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\IMAGE STACK\CONTROL",   # ← REPLACE with your folder path
}

# ┌─────────────────────────────────────────────────────────────────────┐
# │  SHUFFLE MODE                                                       │
# │  "manual"     → you define the exact sequence below                │
# │  "rotation"   → one image per folder, round-robin, random within   │
# │  "weighted"   → folders appear proportionally to their weight      │
# └─────────────────────────────────────────────────────────────────────┘
SHUFFLE_MODE = "rotation"   # ← SET THIS: "manual" | "rotation" | "weighted"

# Used by "rotation" and "weighted" modes
ROUNDS       = 2      # how many full passes through the folder pool
SHUFFLE_SEED = 42     # set to None for a different result each run

# Used by "weighted" mode only — higher number = appears more often
# ┌──────────────────────────────────────────────────────────────────┐
# │  Keys must match the keys in THEME_DIRS exactly                  │
# └──────────────────────────────────────────────────────────────────┘
#WEIGHTS = {
    #"theme_1": 1,   # ← ADJUST weights per theme
    #"theme_2": 1,
    #"theme_3": 2,
    #"theme_4": 1,
    #"theme_5": 1,
#}

# Used by "manual" mode only — list of (folder_key, image_index) tuples
# ┌──────────────────────────────────────────────────────────────────────┐
# │  folder_key must match a key in THEME_DIRS                          │
# │  image_index is the 0-based position within that folder (sorted)    │
# └──────────────────────────────────────────────────────────────────────┘
MANUAL_SEQUENCE = [
    ("SUBSTRATE", 0),   # ← EDIT: (folder_key, image_index)
    ("MACHINE", 0),
    ("CIRCULATION", 0),
    ("REPRESENTATION", 0),
    ("CONTROL", 0),
]

# --- Diffusion params ---
M          = 30
NUM_STEPS  = 8
GUIDANCE   = 0.0
DURATION   = 100
SEED       = 1234
SIZE       = (512, 512)
PROMPT     = "photojournalism, natural light, sharp, unedited, realistic, candid, outdoor, no filter"


## Load Model

In [ ]:
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype = torch.float16,
    variant     = "fp16",
).to(device)
pipe.safety_checker = None

# Expose VAE for latent-space blending
vae = pipe.vae
vae.eval()
print("Model loaded. VAE accessible for latent blending.")


In [ ]:
# --- Latent-space blend utilities ---

def encode_latent(img: "PIL.Image") -> "torch.Tensor":
    """
    Encode a PIL image into the VAE latent space.
    Returns a (1, 4, H/8, W/8) float16 tensor on device.
    """
    import torchvision.transforms.functional as TF
    x = TF.to_tensor(img).unsqueeze(0).to(device, dtype=torch.float16)  # [1,3,H,W] in [0,1]
    x = 2.0 * x - 1.0                                                    # normalize to [-1,1]
    with torch.no_grad():
        latent = vae.encode(x).latent_dist.mean                           # deterministic mean
        latent = latent * vae.config.scaling_factor
    return latent


def decode_latent(latent: "torch.Tensor") -> "PIL.Image":
    """
    Decode a VAE latent tensor back to a PIL image.
    """
    from PIL import Image as PILImage
    import torchvision.transforms.functional as TF
    with torch.no_grad():
        z = latent / vae.config.scaling_factor
        decoded = vae.decode(z).sample                    # [-1,1]
    decoded = (decoded.clamp(-1, 1) + 1.0) / 2.0         # [0,1]
    decoded = decoded.squeeze(0).float().cpu()
    return TF.to_pil_image(decoded)


def latent_blend(img_a: "PIL.Image", img_b: "PIL.Image", t: float) -> "PIL.Image":
    """
    Blend two images by interpolating linearly in VAE latent space.
    t=0 -> img_a, t=1 -> img_b.
    Unlike pixel-space blending, this respects the model's learned
    representation: textures, structures, and semantics blend through
    their encoded content rather than averaging raw RGB values.
    """
    z_a = encode_latent(img_a)
    z_b = encode_latent(img_b)
    z   = (1.0 - t) * z_a + t * z_b     # linear interp in latent space
    return decode_latent(z)


print("Latent blend utilities ready.")


## Core Function: Generate n.gif

Generates a single transition gif from `src` to `tgt` and saves it as `{n}.gif` in `GIFS_DIR`.

The chain: pass the last frame of the previous gif as `src` so drift carries forward naturally.

In [ ]:
def generate_gif(n, src, tgt):
    """
    Generate transition gif from src -> tgt, saved as {n}.gif in GIFS_DIR.

    Blending now happens in the VAE latent space: each intermediate frame
    is produced by interpolating the encoded representations of the previous
    frame and the target, then decoding that blend before passing it into
    the diffusion model. This means the morph flows through the content
    structure of both images rather than averaging raw pixels.

    Parameters
    ----------
    n   : int         - output filename (e.g. 3 -> '3.gif')
    src : PIL.Image   - starting frame (pass last frame of previous gif to chain)
    tgt : PIL.Image   - ending anchor image

    Returns
    -------
    frames : list of PIL.Image   - all frames including src and tgt
    """
    gen    = torch.Generator(device=device).manual_seed(SEED)
    frames = [src]
    prev   = src

    for k in range(1, M + 1):
        lam = k / (M + 1)          # 0 -> 1 (excluding endpoints)
        beta = 0.85 * (lam ** 2.5) # nonlinear easing: slow start, fast finish

        # --- Latent-space blend ---
        # Instead of Image.blend(prev, tgt, beta) which mixes raw RGB values,
        # we encode both images into the VAE's latent space and interpolate there.
        # The decoded result is a frame whose textures and structures are drawn
        # from the model's compressed understanding of both images.
        init = latent_blend(prev, tgt, beta)

        strength = 0.35 + 0.25 * lam   # tops out ~0.60

        img = pipe(
            prompt              = PROMPT,
            image               = init,
            strength            = float(strength),
            guidance_scale      = float(GUIDANCE),
            num_inference_steps = int(NUM_STEPS),
            generator           = gen,
        ).images[0]

        # Color correction
        from PIL import ImageEnhance
        img = ImageEnhance.Color(img).enhance(0.95)
        r, g, b = img.split()
        b = b.point(lambda i: i * 1.02)
        r = r.point(lambda i: i * 0.98)
        img = Image.merge("RGB", (r, g, b))

        frames.append(img)
        prev = img

    frames.append(tgt)

    gif_path = os.path.join(GIFS_DIR, f"{n}.gif")
    frames[0].save(
        gif_path,
        save_all      = True,
        append_images = frames[1:],
        duration      = DURATION,
        loop          = 0,
    )
    print(f"Saved {gif_path}  ({len(frames)} frames)")
    return frames


## Build Deck + Generate All GIFs

Loads images from each theme folder, builds an anchor sequence according to
SHUFFLE_MODE, logs the sequence to a text file, then generates one GIF per
consecutive pair. The last frame of each GIF chains into the next.


In [ ]:
supported = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

themes = {}
for name, path in THEME_DIRS.items():
    files = sorted(
        f for f in os.listdir(path)
        if os.path.splitext(f)[1].lower() in supported
    )
    loaded = []
    skipped = []
    for f in files:
        try:
            img = Image.open(os.path.join(path, f)).convert("RGB").resize(SIZE)
            loaded.append(img)
        except Exception:
            skipped.append(f)
    themes[name] = loaded
    if skipped:
        print(f"  {name}: skipped {len(skipped)} unreadable file(s)")
    print(f"  {name}: {len(loaded)} images loaded")


In [ ]:
import random
rng = random.Random(42)
folder_names = list(themes.keys())
sequence = []
for _ in range(2):
    order = folder_names[:]
    rng.shuffle(order)
    for name in order:
        idx = rng.randint(0, len(themes[name]) - 1)
        sequence.append((name, idx))
anchors = [themes[folder][idx] for folder, idx in sequence]
print(f"Built {len(anchors)} anchors")


In [ ]:
# Generate all gifs in chained order
# last frame of gif N becomes src of gif N+1; wraps back to anchors[0] at the end
all_frames = {}
chain_src  = anchors[0]

for i, tgt in enumerate(anchors[1:] + [anchors[0]], start=1):
    print(f"Generating {i}.gif ...")
    frames = generate_gif(n=i, src=chain_src, tgt=tgt)
    all_frames[i] = frames
    chain_src = frames[-1]   # hand last frame forward

print("\nAll gifs generated.")

In [ ]:
import math
import matplotlib.pyplot as plt

def show_gif_frames_table(all_frames, max_cols=None, figsize_per_cell=(2.2, 2.2), title=True):
    """
    Display frames of each gif (stored in all_frames dict) in a big grid.
    
    all_frames: dict[int, list[PIL.Image]]
        e.g. {1: [img0, img1, ...], 2: [...], ...}
    max_cols: int | None
        If set, cap the number of columns shown (useful if there are lots of frames).
    figsize_per_cell: (w, h)
        Size per cell in inches.
    """
    gif_ids = sorted(all_frames.keys())
    n_gifs = len(gif_ids)
    if n_gifs == 0:
        print("all_frames is empty.")
        return

    # Determine number of columns = max frames across gifs
    max_frames = max(len(all_frames[i]) for i in gif_ids)
    n_cols = min(max_frames, max_cols) if max_cols is not None else max_frames

    # Figure size scales with grid
    fig_w = max(10, n_cols * figsize_per_cell[0])
    fig_h = max(4, n_gifs * figsize_per_cell[1])

    fig, axes = plt.subplots(n_gifs, n_cols, figsize=(fig_w, fig_h))

    # Normalize axes into 2D array indexing [row, col]
    if n_gifs == 1 and n_cols == 1:
        axes = [[axes]]
    elif n_gifs == 1:
        axes = [axes]  # shape -> [1][n_cols]
    elif n_cols == 1:
        axes = [[ax] for ax in axes]  # shape -> [n_gifs][1]

    for r, gif_id in enumerate(gif_ids):
        frames = all_frames[gif_id]
        for c in range(n_cols):
            ax = axes[r][c]
            ax.axis("off")

            if c < len(frames):
                ax.imshow(frames[c])
                ax.set_title(f"gif {gif_id} Â· f{c}", fontsize=8)
            else:
                # blank cell
                ax.set_facecolor("black")

    if title:
        fig.suptitle("All GIFs: frames laid out by (gif row, frame column)", fontsize=14)
        plt.subplots_adjust(top=0.92)

    plt.tight_layout()
    plt.show()


# Use it:
show_gif_frames_table(all_frames, max_cols=None)

## Re-run a Single Gif

If a gif looks bad, re-run just that one. Load the last frame of the previous gif manually to stay in chain.

In [ ]:
# Example: redo gif 3
# Uncomment and adjust as needed

# n = 3
# prev_gif = Image.open(os.path.join(GIFS_DIR, f"{n-1}.gif"))
# prev_gif.seek(prev_gif.n_frames - 1)
# chain_src = prev_gif.copy().convert("RGB")
# tgt = anchors[n % len(anchors)]   # wraps for the last gif
# frames = generate_gif(n=n, src=chain_src, tgt=tgt)
# all_frames[n] = frames

## Display All Gifs in Folder

Reads every `.gif` from `GIFS_DIR` in numeric order and displays them in a scrollable inline grid.
Delete any bad gifs from the folder and re-run this cell to review what remains.

In [ ]:
def combine_gifs(gifs_dir=GIFS_DIR, output_path=r"C:\Users\Calder S. Anderson\Desktop\PARSONS SENIOR YEAR\STUDIO\stabilityaiturboxd_pipeline\Calder\outputs", duration=DURATION):
    """
    Combine all .gif files in gifs_dir into one master gif.

    Parameters
    ----------
    gifs_dir    : str â€” folder containing numbered .gif files
    output_path : str â€” where to save the combined gif
    duration    : int â€” ms per frame in the output gif

    Returns
    -------
    output_path : str â€” path to the saved master gif
    """
    gif_files = sorted(
        [f for f in os.listdir(gifs_dir) if f.endswith(".gif")],
        key=lambda x: int(os.path.splitext(x)[0]) if os.path.splitext(x)[0].isdigit() else 0
    )

    if not gif_files:
        print(f"No gifs found in {gifs_dir!r}")
        return None

    print(f"Combining {len(gif_files)} gifs into {output_path!r} ...")

    all_frames = []

    for i, fname in enumerate(gif_files):
        path = os.path.join(gifs_dir, fname)
        gif  = Image.open(path)

        # Extract all frames from this gif
        frames = []
        try:
            while True:
                frames.append(gif.copy().convert("RGB"))
                gif.seek(gif.tell() + 1)
        except EOFError:
            pass

        # For intermediate gifs, skip first and last frame to avoid duplicates
        # (first frame = last frame of previous gif, last frame = first frame of next gif)
        if i == 0:
            # First gif: keep all frames except the last (it's duplicated in gif 2)
            all_frames.extend(frames[:-1])
        elif i == len(gif_files) - 1:
            # Last gif: skip first frame (duplicated from previous), keep the rest
            all_frames.extend(frames[1:])
        else:
            # Middle gifs: skip both first and last
            all_frames.extend(frames[1:-1])

        print(f"  {fname}: {len(frames)} frames ({len(frames) if i == 0 else len(frames) - 1 if i == len(gif_files) - 1 else len(frames) - 2} kept)")

    # Save combined gif
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    all_frames[0].save(
        output_path,
        save_all      = True,
        append_images = all_frames[1:],
        duration      = duration,
        loop          = 0,
    )

    print(f"\nSaved master gif with {len(all_frames)} total frames to {output_path!r}")
    return output_path


# Run it
master_path = combine_gifs()

# Display the result
if master_path and os.path.exists(master_path):
    display(IPyImage(filename=master_path))

# 22.4s 

In [ ]:
from PIL import Image
import os
from IPython.display import display, Image as IPyImage

GIFS_DIR = "outputs/gifs"
DURATION = 100

master_path = combine_gifs()
if master_path and os.path.exists(master_path):
    display(IPyImage(filename=master_path))